# Deploying GPT-OSS with TensorRT-LLM (AutoDeploy)

This notebook walks you through deploying OpenAI's `openai/gpt-oss-20b` and `openai/gpt-oss-120b` models using TensorRT-LLM's AutoDeploy backend.

[TensorRT-LLM](https://nvidia.github.io/TensorRT-LLM/) is NVIDIA's open-source library for accelerating and optimizing LLM inference on NVIDIA GPUs. AutoDeploy is a one-shot graph-transform pipeline that translates a HuggingFace model into a deploy-ready engine — see the [AutoDeploy guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html) for details.

**Model Resources:**
- [HuggingFace — gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b)
- [HuggingFace — gpt-oss-120b](https://huggingface.co/openai/gpt-oss-120b)
- [OpenAI — gpt-oss announcement](https://openai.com/index/introducing-gpt-oss/)
- [OpenAI — gpt-oss model card](https://openai.com/index/gpt-oss-model-card/)

**Model Highlights:**
- Mixture-of-Experts (MoE) architecture released by OpenAI in 2025 under Apache 2.0
- `gpt-oss-20b`: 21B parameters total, 3.6B active, 24 layers, 32 experts, top-4 routing, ~16 GB MXFP4 weights
- `gpt-oss-120b`: 117B parameters total, 5.1B active, 36 layers, 128 experts, top-4 routing, ~63 GB MXFP4 weights
- Native MXFP4 quantization for MoE weights (handled by AutoDeploy's `quantize_mxfp4_moe` transform)
- 64 query heads / 8 key-value heads (GQA), `head_dim=64`, `hidden_size=2880`
- Per-head learnable attention sinks; alternating sliding (window=128) and full-attention layers
- 131,072 token context length (YaRN-scaled RoPE)
- Channel-based reasoning protocol (`analysis` → `final`)

**Prerequisites:**
- NVIDIA GPU with CUDA 12.x and recent drivers
  - `gpt-oss-20b`: ≥ 1×80 GB GPU (the cookbook below pairs it with 2 GPUs for parallelism)
  - `gpt-oss-120b`: ≥ 4×80 GB or 8×80 GB GPUs (the cookbook below uses 8)
- Python 3.10+
- TensorRT-LLM ([container](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/tensorrt-llm/containers/release) or pip install)

## Prerequisites & Environment

Set up a containerized environment for TensorRT-LLM by running the following command in a terminal:

```shell
docker run --rm -it --ipc=host --ulimit memlock=-1 --ulimit stack=67108864 --gpus=all -p 8000:8000 nvcr.io/nvidia/tensorrt-llm/release:1.3.0rc1
```

You now have TensorRT-LLM set up!

In [ ]:
# If pip not found
!python -m ensurepip --default-pip

In [ ]:
%pip install torch openai

## Verify GPU

Check that CUDA is available and the GPU is detected correctly.

In [ ]:
# Environment check
import sys

import torch

print(f"Python: {sys.version}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Num GPUs: {torch.cuda.device_count()}")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU[{i}]: {torch.cuda.get_device_name(i)}")

## OpenAI-Compatible Server

Start a local OpenAI-compatible server with TensorRT-LLM via the terminal, within the running docker container.

Each gpt-oss size has its own AutoDeploy YAML under `examples/auto_deploy/model_registry/configs/`:
- `gpt_oss_20b.yaml` (world_size=2)
- `gpt_oss_120b.yaml` (world_size=8)

Pick the YAML that matches the model size you want to deploy.

### Load `gpt-oss-20b`

Launch the TensorRT-LLM server on 2 GPUs:

```shell
trtllm-serve "openai/gpt-oss-20b" \
  --host 0.0.0.0 \
  --port 8000 \
  --backend _autodeploy \
  --extra_llm_api_options examples/auto_deploy/model_registry/configs/gpt_oss_20b.yaml
```

### Load `gpt-oss-120b`

Launch the TensorRT-LLM server on 8 GPUs:

```shell
trtllm-serve "openai/gpt-oss-120b" \
  --host 0.0.0.0 \
  --port 8000 \
  --backend _autodeploy \
  --extra_llm_api_options examples/auto_deploy/model_registry/configs/gpt_oss_120b.yaml
```

Both YAMLs are self-contained — they include the compile backend, attention backend, world size, KV-cache settings and the CUDA-graph batch-size buckets needed for serving.

Your server is now running!

## Use the API

Use the OpenAI-compatible client to send requests to the TensorRT-LLM server.

In [ ]:
from openai import OpenAI

# Setup client
BASE_URL = "http://0.0.0.0:8000/v1"
API_KEY = "null"
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Set this to whichever model you started the server with.
MODEL_ID = "openai/gpt-oss-20b"  # or "openai/gpt-oss-120b"

In [ ]:
# Basic chat completion
print("Chat Completion Example")
print("=" * 50)

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Where is the capital of Iceland?"},
    ],
    temperature=1.0,
    top_p=1.0,
    max_tokens=128,
)

print("Response:")
print(response.choices[0].message.content)

In [ ]:
# Streaming chat completion
print("Streaming response:")
print("=" * 50)

stream = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Write a Python function that checks if a number is prime."},
    ],
    temperature=1.0,
    max_tokens=512,
    stream=True,
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="", flush=True)

## Evaluation Parameters

OpenAI's gpt-oss model card recommends the following defaults:

- `temperature`: 1.0
- `top_p`: 1.0
- `max_tokens`: 131072 (model's context limit; trim for serving SLOs)

The model uses a channel-based reasoning protocol (`analysis` → `final`); the `analysis` segment is the model's chain-of-thought and the `final` segment is the user-facing answer. Both are returned via the OpenAI-compatible API as a single response — the chat template handles the formatting.

## Additional Resources

- [TensorRT-LLM Documentation](https://nvidia.github.io/TensorRT-LLM/)
- [AutoDeploy Guide](https://nvidia.github.io/TensorRT-LLM/torch/auto_deploy/auto-deploy.html)
- [gpt-oss-20b on HuggingFace](https://huggingface.co/openai/gpt-oss-20b)
- [gpt-oss-120b on HuggingFace](https://huggingface.co/openai/gpt-oss-120b)
- [OpenAI gpt-oss announcement blog](https://openai.com/index/introducing-gpt-oss/)
- [OpenAI gpt-oss model card](https://openai.com/index/gpt-oss-model-card/)